# Synthetic data generation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def rate(x): return float(np.mean(np.asarray(x)))
def nearest_predict(train_x, train_y, test_x):
    train_x=np.asarray(train_x); train_y=np.asarray(train_y); preds=[]
    for t in np.asarray(test_x):
        preds.append(train_y[int(np.argmin(np.abs(train_x-t)))])
    return np.asarray(preds)
print('setup ok')

## Generating useful examples without changing the question
Synthetic data is useful only when it preserves the structure the model should learn. These examples check moments, class balance, noise, privacy distance, and end-to-end training on generated points.

In [ ]:
# 1) A generator should match basic moments before it is trusted
real=np.array([1.,2.,3.,4.]); synth=np.array([1.2,1.9,3.1,3.8])
plt.figure(figsize=(6,3)); plt.bar(['real mean','synth mean','real sd','synth sd'],[real.mean(),synth.mean(),real.std(),synth.std()]); plt.title('moment check')
assert abs(real.mean()-2.5)<1e-12 and abs(synth.mean()-2.5)<1e-12

In [ ]:
# 2) Conditional generation preserves class balance explicitly
rng=np.random.default_rng(1); y=np.r_[np.zeros(50),np.ones(50)]; x=np.r_[rng.normal(0,1,50),rng.normal(3,1,50)]
plt.figure(figsize=(6,3)); plt.hist(x[y==0],alpha=.6,label='class 0'); plt.hist(x[y==1],alpha=.6,label='class 1'); plt.legend(); plt.title('synthetic classes generated conditionally')
assert y.mean()==0.5

In [ ]:
# 3) Turning up noise increases diversity and risk together
base=np.linspace(0,1,40); noisy=[base+np.random.default_rng(2).normal(0,s,40) for s in [.02,.2]]
plt.figure(figsize=(6,3)); plt.plot(base,noisy[0],label='low noise'); plt.plot(base,noisy[1],label='high noise'); plt.legend(); plt.title('noise knob changes dispersion')
assert noisy[1].std()>noisy[0].std()

In [ ]:
# 4) A privacy check measures nearest-neighbor distance to real records
real=np.array([0.,1.,3.,6.]); synth=np.array([.05,2.2,5.0]); d=np.array([np.min(np.abs(real-s)) for s in synth])
plt.figure(figsize=(6,3)); plt.bar(range(len(synth)),d); plt.axhline(.1,c='r',ls='--'); plt.title('too-close synthetic rows are memorization suspects')
assert abs(d.min()-0.05)<1e-12

In [ ]:
# 5) Synthetic minority points can repair a simple decision boundary
rng=np.random.default_rng(3); neg=rng.normal(0,0.5,40); pos=rng.normal(3,0.5,5); synth_pos=rng.normal(3,0.5,35)
plt.figure(figsize=(6,3)); plt.hist(neg,alpha=.5,label='negative'); plt.hist(np.r_[pos,synth_pos],alpha=.5,label='positive + synthetic'); plt.legend(); plt.title('balanced generated positives')
assert len(np.r_[pos,synth_pos])==len(neg)